# RDKit and pandas

[pandas](https://pandas.pydata.org/) is the standard Python library for working with tables of data. RDKit can put molecules directly into pandas tables, which makes it easy to calculate properties and search large sets of molecules.

This notebook introduces the basics of pandas as we go, so you don't need to have used it before. By the end you should be able to:

- create a table of molecules and add calculated properties to it
- load a data file and select the rows and columns you want
- plot, sort and summarise a column
- run a substructure search on a whole table
- save your results

Cells marked **Exercise** are for you to try. A hidden solution follows each one - click "Solution" to show it, but have a go first!

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import PandasTools
from rdkit.Chem.Draw import IPythonConsole

# draw molecules as pictures in every table
PandasTools.RenderImagesInAllDataFrames(images=True)

It's standard to import pandas as `pd`, so you'll see `pd.` in front of pandas functions.

# 1. A first table

The main object in pandas is the **DataFrame**: a table with named columns and numbered rows, a bit like a spreadsheet. One way to make one is from a dictionary, where each key is a column name and each value is a list containing that column's contents:

In [ ]:
small = pd.DataFrame({
    'name': ['aspirin', 'paracetamol', 'ibuprofen', 'caffeine', 'ethanol'],
    'smiles': ['CC(=O)Oc1ccccc1C(=O)O', 'CC(=O)Nc1ccc(O)cc1', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
               'Cn1cnc2c1c(=O)n(C)c(=O)n2C', 'CCO'],
})
small

The numbers on the left are the **index**, pandas' label for each row.

We get a single column using square brackets and the column name. A column on its own is called a **Series**:

In [ ]:
small['name']

`PandasTools.AddMoleculeColumnToFrame` reads a column of SMILES and adds a new column containing RDKit molecules. These are drawn directly in the table:

In [ ]:
PandasTools.AddMoleculeColumnToFrame(small, smilesCol='smiles', molCol='mol')
small

Now we can calculate a property for every molecule. `.map()` runs a function on each value in a column, and we store the results in a new column by assigning to a new name:

In [ ]:
small['MW'] = small['mol'].map(Descriptors.MolWt)
small

`Descriptors` contains many other property calculators, for example `MolLogP` (lipophilicity), `NumHDonors`, `NumHAcceptors`, `TPSA` (polar surface area), `NumRotatableBonds` and `RingCount`.

To **filter** rows, we write a condition on a column. This gives a True/False value for each row:

In [ ]:
small['MW'] > 190

Putting that condition inside square brackets keeps only the rows where it is True. To select several columns, pass a list of column names, which is why there are double square brackets:

In [ ]:
small[small['MW'] > 190]

In [ ]:
small[['name', 'MW']]

### Exercise 1

1. Add a column called `logP` to `small` using `Descriptors.MolLogP`.
2. Show only the molecules with a logP greater than 1.

In [ ]:
# Exercise 1: your code here

<details>
<summary>Solution</summary>

```python
small['logP'] = small['mol'].map(Descriptors.MolLogP)
small[small['logP'] > 1]
```

You should see aspirin, paracetamol and ibuprofen. Caffeine and ethanol have logP values below zero, meaning they prefer water to octanol.

</details>

# 2. Loading a real dataset

The `data` folder contains a table of drugs downloaded from [ChEMBL](https://www.ebi.ac.uk/chembl/). It is a tab-separated text file (compressed with gzip), which `pd.read_csv` can read directly:

In [ ]:
df = pd.read_csv('../data/chembl_drugs.txt.gz', sep='\t')
df.shape

`shape` gives the number of (rows, columns). That's too big to look at all at once, so we list the column names and use `head()` to see the first 5 rows:

In [ ]:
df.columns

In [ ]:
df[['CHEMBL_ID', 'SYNONYMS', 'DEVELOPMENT_PHASE', 'RULE_OF_FIVE', 'CANONICAL_SMILES']].head()

`value_counts()` counts how often each value appears in a column. `DEVELOPMENT_PHASE` is the clinical trial phase, where 4 means the drug has been approved:

In [ ]:
df['DEVELOPMENT_PHASE'].value_counts()

# 3. Filtering the table

We'll keep only drugs that:

- have a SMILES string (some entries, such as antibodies, don't)
- pass Lipinski's "rule of five", a rule of thumb for drug-like small molecules: MW under 500, logP under 5, at most 5 H-bond donors and at most 10 H-bond acceptors
- are approved (phase 4)

To combine conditions, use `&` for AND and `|` for OR. Each condition must be in brackets:

In [ ]:
has_smiles = df['CANONICAL_SMILES'].notnull()
passes_ro5 = df['RULE_OF_FIVE'] == 'Y'
approved = df['DEVELOPMENT_PHASE'] == 4

drugs = df[has_smiles & passes_ro5 & approved].copy()
len(drugs)

Using `.copy()` makes `drugs` a separate table, so changing it later won't affect (or warn about) the original `df`.

### Exercise 2

The `FIRST_APPROVAL` column gives the year each drug was first approved. How many of our drugs were approved before 1950? Show their `SYNONYMS` and `FIRST_APPROVAL`.

In [ ]:
# Exercise 2: your code here

<details>
<summary>Solution</summary>

```python
old = drugs[drugs['FIRST_APPROVAL'] < 1950]
print(len(old))   # 37
old[['SYNONYMS', 'FIRST_APPROVAL']]
```

</details>

# 4. Adding molecules and names

As before, we add a molecule column. We don't give it a name this time, so RDKit uses the default name `ROMol`:

In [ ]:
PandasTools.AddMoleculeColumnToFrame(drugs, smilesCol='CANONICAL_SMILES')

# .notnull() removes any rows where RDKit couldn't read the SMILES
drugs = drugs[drugs['ROMol'].notnull()]
len(drugs)

## Making short drug names

We'd like to label our drawings with each drug's name. The names are in the `SYNONYMS` column, so let's look at a few:

In [ ]:
drugs['SYNONYMS'].head()

Each entry has the drug's name followed by a list, in brackets, of the organisations that use that name (for example BAN = British Approved Name, FDA = the US Food and Drug Administration). Many drugs also have several alternative names separated by `;`. That's far too long for a label, so we'll keep just the first name: the text before the first `(`.

Let's work out how to do that on one piece of text first, using ordinary Python string methods:

In [ ]:
text = 'Levodopa (BAN, FDA, INN, JAN, USAN, USP)'

pieces = text.split('(')   # split the text at each '(' into a list of pieces
print(pieces)

first = pieces[0]          # keep the first piece: 'Levodopa ' (with a space at the end)
print(first.strip())       # .strip() removes spaces from the start and end

pandas can do the same thing to every row at once. For a column of text, string methods go after `.str`:

- `.str.split('(')` splits every entry into pieces
- `.str[0]` keeps the first piece of each
- `.str.strip()` removes the spaces

We store the result in a new `name` column and compare it with the original:

In [ ]:
drugs['name'] = drugs['SYNONYMS'].str.split('(').str[0].str.strip()

drugs[['SYNONYMS', 'name']].head()

## Missing values

One drug has nothing in its `SYNONYMS` column. pandas marks missing values as `NaN` ("not a number"), and `.isnull()` finds them, just like `.notnull()` in section 3 but the other way round:

In [ ]:
drugs[drugs['name'].isnull()][['CHEMBL_ID', 'SYNONYMS', 'name']]

`.fillna()` replaces missing values with something else. We'll use the drug's ChEMBL ID as its name:

In [ ]:
drugs['name'] = drugs['name'].fillna(drugs['CHEMBL_ID'])

drugs[['name', 'ROMol']].head()

`PandasTools.FrameToGridImage` draws the molecules from a table in a grid. `drugs.head(8)` gives the first 8 rows, and `legendsCol='name'` says which column to use for the labels:

In [ ]:
PandasTools.FrameToGridImage(drugs.head(8), legendsCol='name', molsPerRow=4)

# 5. Calculating and exploring properties

**Descriptors** are numbers calculated from a molecule's structure that describe things like its size or polarity. We'll calculate the four used in Lipinski's rule of five:

- `MolWt` - molecular weight
- `MolLogP` - an estimate of logP, which measures whether a molecule prefers an oily solvent (octanol) or water. Higher means more oily ("lipophilic"), negative means it prefers water.
- `NumHDonors` - number of hydrogen bond donors (roughly, NH and OH groups)
- `NumHAcceptors` - number of hydrogen bond acceptors (mostly N and O atoms)

As in section 1, `.map()` runs the function on every molecule in the column:

In [ ]:
drugs['MW'] = drugs['ROMol'].map(Descriptors.MolWt)
drugs['logP'] = drugs['ROMol'].map(Descriptors.MolLogP)
drugs['HBD'] = drugs['ROMol'].map(Descriptors.NumHDonors)
drugs['HBA'] = drugs['ROMol'].map(Descriptors.NumHAcceptors)

drugs[['name', 'MW', 'logP', 'HBD', 'HBA']].head()

## Summarising a column

A column has methods that summarise it in a single number, such as `.mean()`, `.min()` and `.max()`:

In [ ]:
print('Average MW:', drugs['MW'].mean())
print('Smallest MW:', drugs['MW'].min())
print('Largest MW:', drugs['MW'].max())

`describe()` calculates several of these at once, for each column:

- `count` - how many values there are
- `mean` and `std` - the average and standard deviation (how spread out the values are)
- `min` and `max` - the smallest and largest values
- `25%`, `50%`, `75%` - the quartiles. For example, a quarter of the drugs have values below `25%`. The `50%` value is the median.

In [ ]:
drugs[['MW', 'logP', 'HBD', 'HBA']].describe()

No drug has more than 5 H-bond donors, 10 acceptors or a MW above 500, as expected since we kept only drugs that pass the rule of five.

## Plotting

pandas can plot a column directly. A **histogram** splits the values into ranges ("bins") and counts how many drugs fall into each one. `bins=30` sets the number of bars.

The plotting function gives back the plot (we call it `ax`), which we can use to add axis labels. The `;` at the end of the last line stops Jupyter printing some extra text above the plot.

In [ ]:
ax = drugs['MW'].hist(bins=30)
ax.set_xlabel('Molecular weight')
ax.set_ylabel('Number of drugs');

Most drugs have a MW between about 200 and 400, and there are none above 500 because of the rule-of-five filter.

A **scatter plot** compares two columns. Each point is one drug. `alpha=0.3` makes the points see-through, so you can see where lots of them overlap:

In [ ]:
drugs.plot.scatter(x='MW', y='logP', alpha=0.3);

Larger molecules tend to have a higher logP, but there's a lot of spread: a large molecule with many polar groups can still prefer water.

## Sorting

`sort_values` sorts the table by a column. Use `ascending=False` to put the largest values first. The line below does three things, reading left to right: sort by logP, keep three columns, and show the first 5 rows.

In [ ]:
drugs.sort_values('logP', ascending=False)[['name', 'MW', 'logP']].head()

Notice the numbers on the left: each row keeps its original index when the table is sorted or filtered.

### Exercise 3

1. Make a histogram of logP with labelled axes.
2. Draw the 6 **smallest** drugs by molecular weight. Any surprises?
3. The table was filtered to drugs that pass the rule of five, so how many have an RDKit logP above 5?

In [ ]:
# Exercise 3: your code here

<details>
<summary>Solution</summary>

```python
ax = drugs['logP'].hist(bins=30)
ax.set_xlabel('logP')
ax.set_ylabel('Number of drugs');
```

```python
smallest = drugs.sort_values('MW').head(6)
PandasTools.FrameToGridImage(smallest, legendsCol='name')
```

They are ammonia, water, nitrogen, nitric oxide, oxygen and ethanol, all listed as approved medicines!

```python
len(drugs[drugs['logP'] > 5])   # 28
```

logP is a calculated estimate, and different programs give different values. ChEMBL's rule-of-five flag uses a different logP calculation from RDKit's, so a few drugs are just over 5 here.

</details>

# 6. Substructure searching a table

PandasTools lets us run a substructure search on a whole molecule column using `>=`, which you can read as "contains". The result is True/False for each row, so it can be used as a filter just like before.

Here we search for the core of the sulfonamide ("sulfa") drugs, an aniline with a sulfonyl group on the opposite side of the ring:

In [ ]:
sulfa = Chem.MolFromSmarts('N-c1ccc(-S(=O)(=O)-[*])cc1')
sulfa

In [ ]:
sulfa_drugs = drugs[drugs['ROMol'] >= sulfa]
len(sulfa_drugs)

The matching part is highlighted in the table:

In [ ]:
sulfa_drugs[['name', 'ROMol']].head()

For a grid image, we pass the matching atoms with `highlightAtomLists`, as in notebook 2:

In [ ]:
first12 = sulfa_drugs.head(12)

PandasTools.FrameToGridImage(first12, legendsCol='name', molsPerRow=4,
                             highlightAtomLists=[mol.GetSubstructMatch(sulfa) for mol in first12['ROMol']])

As well as sulfa antibiotics (sulfadiazine, sulfapyridine...), the pattern also finds thiazide diuretics such as hydrochlorothiazide, which contain the same arrangement of atoms inside a ring.

We can also save the search result as a new True/False column. Using the carboxylic acid pattern from notebook 2:

In [ ]:
acid = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')
drugs['has_acid'] = drugs['ROMol'] >= acid

drugs['has_acid'].value_counts()

`groupby` splits the table into groups by the values in one column, then summarises another column for each group. Here's the average logP of drugs with and without a carboxylic acid:

In [ ]:
drugs.groupby('has_acid')['logP'].mean()

Drugs with an acid group have a lower average logP, i.e. they are more polar, which makes chemical sense.

### Exercise 4

Do the same for halogens (`[F,Cl,Br,I]`): add a `has_halogen` column, count how many drugs contain a halogen, and compare the average logP with and without one.

In [ ]:
# Exercise 4: your code here

<details>
<summary>Solution</summary>

```python
halogen = Chem.MolFromSmarts('[F,Cl,Br,I]')
drugs['has_halogen'] = drugs['ROMol'] >= halogen

print(drugs['has_halogen'].value_counts())      # 338 True
drugs.groupby('has_halogen')['logP'].mean()     # about 1.9 without, 2.8 with
```

Halogenated drugs are more lipophilic on average.

</details>

### Exercise 5 (stretch)

The `INDICATION_CLASS` column says what each drug is used for. What are the three most common indication classes among the drugs that contain a carboxylic acid?

In [ ]:
# Exercise 5: your code here

<details>
<summary>Solution</summary>

```python
acid_drugs = drugs[drugs['has_acid']]
acid_drugs['INDICATION_CLASS'].value_counts().head(3)
```

Antibacterial, anti-inflammatory and antihypertensive. Many antibiotics (e.g. penicillins) and anti-inflammatories (e.g. ibuprofen) contain a carboxylic acid.

</details>

# 7. Saving results

To save a table as a CSV file (which you can open in Excel), choose the columns you want and use `to_csv`. `index=False` leaves out the row numbers. The molecule column can't be saved in a CSV, but the SMILES column can:

In [ ]:
columns = ['CHEMBL_ID', 'name', 'CANONICAL_SMILES', 'MW', 'logP', 'has_acid']
drugs[columns].to_csv('drug_properties.csv', index=False)

To keep the molecules, save as an SD file instead. The `approved_drugs.sdf` file used in notebook 2 was made like this. Here we save just the sulfa drugs to a new file:

In [ ]:
PandasTools.WriteSDF(sulfa_drugs, 'sulfa_drugs.sdf', idName='CHEMBL_ID', properties=['name', 'MW', 'logP'])

You can read an SD file back into a table with `PandasTools.LoadSDF`:

In [ ]:
PandasTools.LoadSDF('sulfa_drugs.sdf').head()

Tutorial originally by Samo Turk, Jan. 2017. Expanded for this workshop.